In [1]:
#import libraries 
import torch 
import torch.nn as nn 
import torch.optim as optim
from matplotlib import pyplot as plt 

In [2]:
t_c = [0.5,  14.0, 15.0, 28.0, 11.0,  8.0, 3.0, -4.0, 6.0, 13.0, 21.0] # temperature in celsius
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4] #temperature in unknown unit
t_c = torch.tensor(t_c).unsqueeze(1)
t_u = torch.tensor(t_u).unsqueeze(1)

# spliting data
n_samples = t_u.shape[0]
n_val = int(0.2 * n_samples)
shuffled_indices = torch.randperm(n_samples)
train_indices = shuffled_indices[:-n_val]
val_indices = shuffled_indices[-n_val:]
train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]
val_t_c = t_c[val_indices]
val_t_u = t_c[val_indices]
train_t_un = 0.1 * train_t_u
val_t_un = 0.1 * val_t_u 

In [3]:
"""
All PyTorch-provided subclasses of nn.Module have their __call__ method defined.
This allows us to instantiate an nn.Linear and call it as if it was a function, like so
"""
linear_model = nn.Linear(1, 1) #constructor ags It accepts three args. input features, the number of output features and wheather the linear model should include bias or not (deafault is True)
print(linear_model(val_t_un))
print(linear_model.weight)
print(linear_model.bias)

tensor([[0.5207],
        [0.2711]], grad_fn=<AddmmBackward>)
Parameter containing:
tensor([[-0.1314]], requires_grad=True)
Parameter containing:
tensor([0.4682], requires_grad=True)


In [4]:
#testing the linear model on a dataset

"""
Although PyTorch lets us get away with it, we don’t actually provide an input with the right dimensionality. We have a model that takes one input and produces one output
but PyTorch nn.Module and its subclasses are designed to do so on multiple samples at the same time. 
To accommodate multiple samples, modules expect the zeroth dimension of the input to be the number of samples in the batch.
"""

x = torch.ones(1) 
print(x)
linear_model(x),

tensor([1.])


(tensor([0.3368], grad_fn=<AddBackward0>),)

In [5]:
# Batching inputs
"""
Any module in nn is wirtten to produce outputs for a batch of multiple inputs at the same time.
Thus assuming we need to run nn.Linear on 10 samples, we can create an input tensor of size B x Nin where B=size of Batch and Nin is the number of input features and run it once thorught the model.,
Eg. below 
"""
x = torch.ones(10,1)
print(x)
linear_model(x)

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])


tensor([[0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368],
        [0.3368]], grad_fn=<AddmmBackward>)

In [7]:
# optimized batching
# update the leinear model and passing it to the optimizer
linear_model = nn.Linear(1, 1)
optimizer = optim.SGD(linear_model.parameters(), lr=1e-2) 

# creating a training loop
def training_loop(n_epochs, optimizer, model, loss_fn, train_t_u, val_t_u, train_t_c, val_t_c):
    for epoch in range(1, n_epochs+1):
        train_t_p = model(train_t_u)
        loss_train = loss_fn(train_t_p, train_t_c) 

        val_t_p = model(val_t_u) 
        loss_val = loss_fn(val_t_p, val_t_c) 

        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step() 

        if epoch == 1 or epoch%1000 == 0:
            print(f"Epoch {epoch}, Training loss {loss_train.item():.4f}",
                    f"Validation loss {loss_val.item():.4f}")





# Using model params above on trainin_loop 
training_loop(n_epochs=3000, optimizer=optimizer, model=linear_model, loss_fn=nn.MSELoss(), train_t_u=train_t_un, val_t_u=val_t_un, train_t_c=train_t_c, val_t_c=val_t_c)
print()
print(linear_model.weight)
print(linear_model.bias)

Epoch 1, Training loss 163.5342 Validation loss 111.9526
Epoch 1000, Training loss 4.9890 Validation loss 292.2341
Epoch 2000, Training loss 3.1062 Validation loss 418.3820
Epoch 3000, Training loss 2.9513 Validation loss 459.6381

Parameter containing:
tensor([[5.5515]], requires_grad=True)
Parameter containing:
tensor([-18.5723], requires_grad=True)


In [ ]:
# Replacing the linear model 
